# Exploratory Data Analysis: WANDS Dataset

This notebook explores Wayfair's WANDS (Wayfair ANnotation DataSet) to understand:
1. Product feature distribution and coverage
2. Attribute value frequencies for training label quality
3. Data quality issues to address in preprocessing

Reference: [WANDS GitHub](https://github.com/wayfaireng/WANDS)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from src.data.wands_loader import WANDSLoader
from src.data.feature_parser import (
    parse_feature_string,
    normalize_attributes,
    process_products_dataframe,
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries loaded successfully')

## 1. Load WANDS Dataset

In [ ]:
# Update this path to your WANDS download location
DATA_DIR = '../data/raw/WANDS/dataset'

loader = WANDSLoader(DATA_DIR)
products = loader.products
queries = loader.queries
labels = loader.labels

print(f'Products: {len(products):,}')
print(f'Queries: {len(queries):,}')
print(f'Labels: {len(labels):,}')
print(f'\nProduct columns: {list(products.columns)}')

In [ ]:
# Sample products
products.head(3)

## 2. Feature Coverage Analysis

In [ ]:
# What fraction of products have features?
has_features = products['product_features'].notna() & (products['product_features'].str.strip() != '')
print(f'Products with features: {has_features.sum():,} / {len(products):,} ({has_features.mean()*100:.1f}%)')

# Distribution of feature count per product
feature_counts = products.loc[has_features, 'product_features'].apply(
    lambda x: len(x.split('|')) if isinstance(x, str) else 0
)

fig, ax = plt.subplots()
feature_counts.hist(bins=20, ax=ax, edgecolor='black')
ax.set_xlabel('Number of Features per Product')
ax.set_ylabel('Count')
ax.set_title('Distribution of Feature Count per Product')
plt.tight_layout()
plt.show()

print(f'\nMedian features per product: {feature_counts.median():.0f}')
print(f'Mean features per product: {feature_counts.mean():.1f}')

## 3. Parse and Analyze Attributes

In [ ]:
# Parse all features
products_with_features = loader.get_products_with_features()
processed = process_products_dataframe(products_with_features)
processed.head()

In [ ]:
# Attribute coverage
attrs = ['style', 'primary_material', 'color_family', 'room_type', 'product_type', 'assembly_required']

coverage = {}
for attr in attrs:
    non_null = processed[attr].notna().sum()
    coverage[attr] = non_null / len(processed) * 100

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(coverage.keys(), coverage.values(), color='steelblue', edgecolor='black')
ax.set_ylabel('Coverage (%)')
ax.set_title('Attribute Coverage in WANDS Products')
ax.set_ylim(0, 100)
for bar, pct in zip(bars, coverage.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{pct:.0f}%', ha='center')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Top values for each attribute
for attr in ['style', 'primary_material', 'color_family']:
    values = processed[attr].dropna()
    top = values.value_counts().head(10)
    print(f'\n=== {attr.upper()} (top 10) ===')
    for val, count in top.items():
        print(f'  {val}: {count} ({count/len(values)*100:.1f}%)')

## 4. Product Class Distribution

In [ ]:
# Top product classes
top_classes = products['product_class'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 6))
top_classes.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Count')
ax.set_title('Top 20 Product Classes in WANDS')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f'Total unique product classes: {products["product_class"].nunique()}')

## 5. Key Takeaways for Model Training

**Data strengths:**
- Large product catalog with real Wayfair data
- Structured features provide clean training labels
- Diverse product categories

**Challenges to address:**
- Not all attributes have equal coverage
- Long-tail distribution of attribute values
- Style is subjective and has the most ambiguity
- Some products have minimal feature information